# Dokumentacja: Generowanie zbioru danych do oceny pozycji szachowych

Źródło danych: Lichess, czerwiec 2025 (~93M partii).
Cel: zbudować zrównoważony zbiór pozycji z ocenami silnika, bez przeciążania pamięci lub CPU.

## 1. Zakres i założenia

Celem jest przygotowanie dużego zbioru danych (np. 1M pozycji) zawierającego:
- FEN pozycji
- Ocenę Stockfish (cp + znormalizowana + flaga mata)
- Faza gry (OTWARCIE / ŚRODEK / KONIEC)
- Podstawowe cechy materiałowe
- Metadane techniczne (czas oceny, głębokość)

Ograniczenia:
- Nie ładować całego PGN do pamięci
- Utrzymać niski koszt silnika
- Zachować równowagę faz
- Unikać trywialnych duplikatów

---

## 2. Problem: Skala danych

Wejście: ~93M partii → nie jest możliwe:
- Jednoczesne ładowanie całego pliku
- Pełna dekompresja archiwum .zst na dysk

Rozwiązanie:
- Strumieniowe parsowanie PGN (`chess.pgn.read_game(stream)`)
- Obsługa `.zst` przez:
  - Pythonowy moduł `zstandard` (preferowany)
  - Alternatywnie zewnętrzny proces `zstd -dc`

---

## 3. Problem: Koszt oceny silnika

Ocena każdej pozycji jest zbyt kosztowna.

Rozwiązania:
- Próbkowanie (tylko wybrane pozycje)
- Ocena w partiach + procesy równoległe
- Bardzo krótki czas na pozycję (szybka ocena, np. 12 ms)
- Ograniczenie zakresu oceny do [-2000, 2000]

---

## 4. Problem: Wybór pozycji (sampling)

Cel: informatywne i taktyczne pozycje bez przeładowania zbioru otwarciami.

Zasady:
- Rozważ pozycję jeśli:
  - `ply % 4 == 0` (lub %2 w trybie skupienia na końcówkach), LUB
  - ruch jest biciem, LUB
  - ruch jest promocją
- Limit na partię: `MAX_POSITIONS_PER_GAME` (np. 8)

Uzasadnienie:
- Zapobiega dominacji długich partii
- Uchwycenie taktycznych/materialnych przejść

---

## 5. Problem: Identyfikacja fazy gry

Numer posunięcia jest niewiarygodny.

Heurystyka:
- `phase_raw = 4*Q + 2*R + (B + N)`; normalizacja przez 24
- KONIEC jeśli:
  - brak hetmanów, LUB
  - suma figur+pionów ≤ 10 (bez królów), LUB
  - brak ciężkich figur i ≤ 1 lekka figura
- W przeciwnym razie: OTWARCIE jeśli `phase_norm ≥ 0.66`, w innym wypadku ŚRODEK

Cel: zbalansować zbiór wg zadanych celów.

---

## 6. Problem: Niedoreprezentowanie końcówek

Końcówki są naturalnie rzadkie.

Mechanizm:
- Planowany udział = `target_end / total_target`
- Jeśli `(planowany - aktualny) > ENDGAME_FOCUS_TRIGGER_GAP` → włącz skupienie na końcówkach
- W skupieniu: gęstsze próbkowanie (co 2 ply)
- Auto wyłączenie po wyrównaniu

---

## 7. Problem: Duplikaty pozycji

Wiele powtarzających się struktur (otwarcia, techniczne końcówki).

Strategia:
- Zredukowany FEN: pierwsze 4 pola (rozmieszczenie figur, ruch, prawa do roszady, pole en passant)
- Pomijaj licznik półruchów i numer pełnego ruchu
- Utrzymuj zbiór `seen_fen`

Uzasadnienie:
- Unikanie sztucznej różnorodności z liczników
- Redukcja nadreprezentacji identycznych struktur z początku partii

---

## 8. Problem: Przepustowość i koordynacja

Potrzebny wydajny przepływ:
- Zbieraj metadane → `pending`
- Gdy partia pełna → równoległa ocena → wyniki do bufora sharda
- Gdy shard pełny → zapis do CSV

Korzyści:
- Niższy narzut oceny
- Odporność na awarie (tracimy tylko niezapisany bufor)

---

## 9. Problem: Normalizacja ocen

Surowe oceny w centypawnach mogą być ekstremalne.

Rozwiązanie:
- Ograniczenie do [-2000, 2000]
- `eval_norm = eval_cp / 2000`
- Maty mapowane na ±2000 + osobna flaga `mate_flag`

Cel: stabilny zakres liczbowy dla modeli uczenia.

---

## 10. Problem: Jeden wielki plik wyjściowy

Problemy:
- Ryzyko (uszkodzenie, przerwanie)
- Trudny w transferze
- Ciężki do otwarcia

Rozwiązanie:
- Sharding (np. co 100k pozycji)
- Pliki nazwane `positions_shard_0001.csv` itd.
- Opróżnienie pozostałości przy finalizacji

---

## 11. Problem: Błędy i przerwania

Potencjalne awarie:
- Uszkodzone segmenty PGN
- Wyjątki silnika
- Przerwanie przez użytkownika (Ctrl+C)
- Brak narzędzi do dekompresji

Zapobieganie:
- `try/except` przy parsowaniu PGN (logowanie + kontynuacja)
- Awaria silnika → neutralna ocena (0 cp, głębokość -1)
- Obsługa SIGINT wywołuje `finalize()`
- Sprawdzenie dostępności `zstandard` / `zstd`

---

## 12. Problem: Elastyczność parametrów

Różne środowiska → różne ograniczenia.

Konfigurowalne przez CLI:
- Cele faz (`--target-open` itd.) lub `--target-total` (podział 30/30/40)
- Rozmiar partii
- Liczba instancji silnika
- Czas na pozycję
- Rozmiar hasha
- Wyłączenie skupienia na końcówkach
- Ograniczenie liczby partii (`--max-games`) do testów

Kompromisy:
- Zbyt wiele instancji > rdzenie CPU → spowolnienie
- Za mała partia → duży narzut; za duża → wolniejsze sprzężenie zwrotne

---

## 13. Problem: Zachowanie kolejności wyników

Równoległe procesy zwracają wyniki w różnej kolejności.

Rozwiązanie:
- Dodanie indeksu do każdego zadania
- Odtworzenie listy wyników przez przypisanie do slotów wg indeksu

---

## 14. Problem: Rozszerzalność

Przyszłe potrzeby:
- „Złoty podzbiór” z większą głębokością
- Dodatkowe cechy (hash struktury pionowej, mobilność)
- Agregacja wielomiesięczna

Projekt zachowany w formie modułowej dla łatwego rozszerzania.

---

## 15. Potencjalne rozszerzenia (roadmap)

- Złoty podzbiór z głębszą oceną / MultiPV
- Eksport do Parquet
- Sampling na podstawie zmian ocen (przed vs po ruchu)
- Statystyki powodów próbkowania (bicie vs interwał)
- Inżynieria cech pozycji (bezpieczeństwo króla, wyspy pionowe)
- Rozproszone przetwarzanie wielowęzłowe

---

## 16. Walidacja po generacji

Zalecane kontrole:
- Liczby docelowe (OTWARCIE/ŚRODEK/KONIEC) się zgadzają
- Widoczna aktywacja skupienia na końcówkach w logach gdy jest niedobór
- Zero (lub znikome) duplikaty po zredukowanym FEN
- Rozkład `eval_cp` (brak dominujących ekstremów)
- Ręczna inspekcja losowych pozycji końcowych

---

## 17. Przegląd przepływu

```
strumień PGN -> parsuj partię -> iteruj ruchy
  -> decyzja samplingowa -> klasyfikuj fazę -> deduplikacja
    -> buduj PositionMeta -> próg partii?
       -> równoległa ocena silnika -> lista EvalResult
          -> dodaj do bufora sharda -> pełny shard? zapisz CSV
finalizacja: opróżnij pending, zapisz pozostały shard
```

---

## 18. Przykład uruchomienia (skala testowa)

```
python prepare_dataset.py \
  --pgn lichess_db_standard_rated_2025-06.pgn.zst \
  --stockfish /usr/bin/stockfish \
  --out-dir data_shards \
  --target-total 50000 \
  --batch-eval-size 400 \
  --engine-instances 6 \
  --engine-time 0.010
```

---

## 19. Tabela porównawcza podsumowania

| Problem | Technika | Powód |
|---------|----------|-------|
| Skala | Strumieniowanie | Niskie użycie pamięci |
| Rzadkie końcówki | Adaptacyjne skupienie | Zbalansowany rozkład faz |
| Koszt silnika | Szybka ocena + równoległość | Przepustowość |
| Duplikaty | Zredukowany FEN | Autentyczna różnorodność |
| Ryzyko przerwania | Sharding + finalize | Minimalna strata |
| Równowaga faz | Cele + monitoring | Kontrolowany skład zbioru |

---

## 20. Podsumowanie

To rozwiązanie obejmuje:
- Wielkoskalowe przetwarzanie (strumieniowanie)
- Efektywną ocenę (partie + multiprocessing)
- Równowagę faz (cele + adaptacyjne skupienie na końcówkach)
- Różnorodność (reguły samplingowe + deduplikacja)
- Stabilność (obsługa błędów, sygnałów)
- Praktyczne użycie (shardowane CSV, znormalizowane oceny)

Projekt przygotowany tak, by umożliwiać iteracyjne ulepszanie bez konieczności przepisywania od zera.